# VeriSieve Track C5 — Graph-Grounded Packed LoRA (`T2` / `T3`) Ablation (Colab T4 / L4)

> **Zero-Egress Privacy Invariant:** Run VeriSieve locally first (`node scripts/export-graph-sft-corpus.mjs --corpus <dir>`) so ONLY strictly redacted `KEEP_SAFE` salted-pseudonym records (`<PII:PERSON_01>`) from public or pre-redacted corpora are uploaded here. Never upload raw unredacted private PDFs.

This notebook evaluates whether Track C's **Graph-Grounded Packed SFT + Single Clean EOS** eliminates the **90% immediate-EOS collapse (45/50 blank answers)** measured on old flat 49.4-token constant-prompt LoRA runs.

In [ ]:
!pip -q install -U "trl>=0.12.0" "peft>=0.13.0" "datasets>=3.0.0" "transformers>=4.46.0" "accelerate>=1.0.0"
import torch
print('PyTorch:', torch.__version__, '| CUDA:', torch.cuda.is_available(), '| GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

In [ ]:
# Upload your locally VeriSieve-redacted, 100%-unique rtio-graph-sft.jsonl
import os
DATA_PATH = 'rtio-graph-sft.jsonl'
if not os.path.exists(DATA_PATH):
    try:
        from google.colab import files
        uploaded = files.upload()
        DATA_PATH = next(iter(uploaded.keys()))
    except ImportError:
        raise FileNotFoundError('Place rtio-graph-sft.jsonl in the working directory.')
print('Loaded dataset:', DATA_PATH, '| size bytes:', os.path.getsize(DATA_PATH))

In [ ]:
# Configuration — switch MODEL_ID to 'Qwen/Qwen2.5-0.5B-Instruct' or 'Qwen/Qwen2.5-1.5B-Instruct' as desired
MODEL_ID = 'LiquidAI/LFM3-1.7B-Base'
MAX_SEQ_LENGTH = 1024
LORA_R = 16
LORA_ALPHA = 32
EPOCHS = 1
LEARNING_RATE = 2e-4
OUTPUT_DIR = './verisieve-track-c-lora-out'

import json, re, inspect
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForCausalLM

SPECIAL_TRAILING_EOS = ('<|im_end|>', '<|endoftext|>', '<|eot_id|>', '<|end|>', '</s>')
CANDIDATE_TARGETS = ('q_proj','k_proj','v_proj','o_proj','out_proj','in_proj','gate_proj','up_proj','down_proj','w1','w2','w3','fc1','fc2')
RAW_PII_PATTERNS = (
    re.compile(r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b'),
    re.compile(r'\b\d{3}-\d{2}-\d{4}\b'),
)
CARD_PATTERN = re.compile(r'\b(?:\d{4}[-\s]?){3}\d{4}\b')
def is_luhn_card(raw):
    d = [int(c) for c in raw if c.isdigit()]
    s = ''.join(str(x) for x in d)
    if len(d) < 13 or len(d) > 19 or not (s.startswith('4') or s.startswith(('51','52','53','54','55','34','37','6011','65')) or (2221 <= int(s[:4]) <= 2720)): return False
    chk, par = 0, len(d) % 2
    for i, v in enumerate(d):
        if i % 2 == par: v = v * 2 - 9 if v * 2 > 9 else v * 2
        chk += v
    return chk % 10 == 0

def strip_trailing_eos(text, eos_token=None):
    cleaned = str(text or '').rstrip()
    markers = list(SPECIAL_TRAILING_EOS)
    if eos_token and eos_token not in markers:
        markers.insert(0, eos_token)
    changed = True
    while changed and cleaned:
        changed = False
        for m in markers:
            if m and cleaned.endswith(m):
                cleaned = cleaned[:-len(m)].rstrip()
                changed = True
    return cleaned

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.padding_side = 'right'
if tokenizer.pad_token is None:
    if tokenizer.unk_token and tokenizer.unk_token != tokenizer.eos_token:
        tokenizer.pad_token = tokenizer.unk_token
    else:
        tokenizer.pad_token = tokenizer.eos_token

raw_rows = [json.loads(line) for line in Path(DATA_PATH).read_text(encoding='utf-8').splitlines() if line.strip()]
has_template = bool(getattr(tokenizer, 'chat_template', None))

formatted_rows = []
seen_texts = set()
raw_pii_hits = 0
double_eos = 0
token_lens = []

for rec in raw_rows:
    if rec.get('mode') == 'packed_passage' and rec.get('text'):
        txt = rec['text']
    elif isinstance(rec.get('messages'), list):
        if has_template:
            txt = tokenizer.apply_chat_template(rec['messages'], tokenize=False, add_generation_prompt=False)
        else:
            txt = '\n\n'.join(f"### {m['role'].capitalize()}\n{m['content']}" for m in rec['messages'])
    else:
        txt = rec.get('text') or rec.get('raw_clean_text') or ''
    txt = strip_trailing_eos(txt, tokenizer.eos_token).strip()
    if not txt or txt in seen_texts:
        continue
    seen_texts.add(txt)
    for pat in RAW_PII_PATTERNS:
        raw_pii_hits += len(pat.findall(txt))
    for m in CARD_PATTERN.finditer(txt):
        if is_luhn_card(m.group(0)): raw_pii_hits += 1
    ids = tokenizer.encode(txt, add_special_tokens=False) + [tokenizer.eos_token_id]
    token_lens.append(len(ids))
    for i in range(len(ids) - 1):
        if ids[i] == tokenizer.eos_token_id and ids[i + 1] == tokenizer.eos_token_id:
            double_eos += 1
    formatted_rows.append({'text': txt})

print(json.dumps({
    'unique_rows': len(formatted_rows),
    'mean_tokens_per_row': round(sum(token_lens) / max(1, len(token_lens)), 1),
    'double_eos_count': double_eos,
    'raw_pii_hits': raw_pii_hits,
}, indent=2))
assert double_eos == 0, 'EOS invariant violated!'
assert raw_pii_hits == 0, 'Raw PII detected in training shard!'

In [ ]:
# Train T2: Graph-Grounded Packed LoRA (packing=True, single clean EOS)
import time
from datasets import Dataset
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

dtype = torch.bfloat16 if (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) else torch.float16
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, device_map='auto', torch_dtype=dtype)

detected = sorted({name.rsplit('.', 1)[-1] for name, _ in model.named_modules() if name.rsplit('.', 1)[-1] in CANDIDATE_TARGETS})
print('Auto-detected LoRA target modules:', detected)

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=0.05,
    target_modules=detected,
    bias='none',
    task_type='CAUSAL_LM',
)

sft_sig = inspect.signature(SFTConfig.__init__).parameters
sft_kwargs = {
    'output_dir': OUTPUT_DIR,
    'num_train_epochs': EPOCHS,
    'per_device_train_batch_size': 1,
    'gradient_accumulation_steps': 4,
    'learning_rate': LEARNING_RATE,
    'logging_steps': 5,
    'save_strategy': 'epoch',
    'report_to': 'none',
    'bf16': dtype == torch.bfloat16,
    'fp16': dtype == torch.float16,
}
if 'dataset_text_field' in sft_sig: sft_kwargs['dataset_text_field'] = 'text'
if 'max_seq_length' in sft_sig: sft_kwargs['max_seq_length'] = MAX_SEQ_LENGTH
elif 'max_length' in sft_sig: sft_kwargs['max_length'] = MAX_SEQ_LENGTH
if 'packing' in sft_sig: sft_kwargs['packing'] = True

trainer_sig = inspect.signature(SFTTrainer.__init__).parameters
trainer_kwargs = {
    'model': model,
    'args': SFTConfig(**sft_kwargs),
    'train_dataset': Dataset.from_list(formatted_rows),
    'peft_config': lora_config,
}
if 'processing_class' in trainer_sig: trainer_kwargs['processing_class'] = tokenizer
else: trainer_kwargs['tokenizer'] = tokenizer

if torch.cuda.is_available(): torch.cuda.reset_peak_memory_stats()
t0 = time.perf_counter()
trainer = SFTTrainer(**trainer_kwargs)
train_out = trainer.train()
train_sec = round(time.perf_counter() - t0, 2)
peak_vram_gb = round(torch.cuda.max_memory_allocated() / (1024**3), 2) if torch.cuda.is_available() else 0.0
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print({'train_sec': train_sec, 'train_loss': round(float(train_out.training_loss), 4), 'peak_vram_gb': peak_vram_gb})

In [ ]:
# Evaluate Blank / Immediate-EOS Collapse Rate (Target: 0.0% vs 90% old flat-line LoRA baseline)
FROZEN_PROBES = [
    ('single_hop', 'How is an APP entity defined under s 6(1) of the Privacy Act in the OAIC APP Guidelines?', [r'agency', r'organi[sz]ation']),
    ('single_hop', 'What are the four key elements of valid consent under the OAIC APP Guidelines?', [r'informed', r'voluntar', r'current', r'capacity']),
    ('single_hop', 'Under the Privacy Act s 6(1), does personal information cover deceased persons?', [r'not\s+include|does\s+not|natural\s+person|living']),
    ('numeric_threshold', 'What is the annual turnover threshold for a small business operator under s 6D of the Privacy Act?', [r'3,000,000|3\s*million']),
    ('numeric_threshold', 'On what exact date do the APP 1.7-1.9 automated decision-making transparency obligations come into effect?', [r'10\s+december\s+2026|december\s+10,?\s+2026']),
    ('numeric_threshold', 'When assessing consent capacity under 18, at what age may an APP entity presume capacity if case-by-case assessment is impracticable?', [r'\b15\b']),
]

model.eval()
eval_rows = []
for cat, q, pats in FROZEN_PROBES:
    prompt = f'### User\n{q}\n\n### Assistant\n'
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    with torch.no_grad():
        out_ids = model.generate(**inputs, max_new_tokens=128, do_sample=False, pad_token_id=tokenizer.pad_token_id, eos_token_id=tokenizer.eos_token_id)
    gen = tokenizer.decode(out_ids[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()
    is_blank = len(gen) < 5
    hits = sum(1 for p in pats if re.search(p, gen, re.I))
    eval_rows.append({'category': cat, 'query': q, 'is_blank': is_blank, 'pattern_hits': f'{hits}/{len(pats)}', 'answer': gen})

blank_rate = round(100.0 * sum(1 for r in eval_rows if r['is_blank']) / len(eval_rows), 1)
report = {
    'model': MODEL_ID,
    'blank_eos_collapse_rate_pct': blank_rate,
    'peak_vram_gb': peak_vram_gb,
    'train_sec': train_sec,
    'rows': eval_rows,
}
Path('track-c-colab-ablation-results.json').write_text(json.dumps(report, indent=2))
print(json.dumps({k: v for k, v in report.items() if k != 'rows'}, indent=2))
for r in eval_rows:
    print(f"[{r['category']}] blank={r['is_blank']} hits={r['pattern_hits']} -> {r['answer'][:140]}")